<a href="https://colab.research.google.com/github/PriyanshuBhunia/classification-ML/blob/main/CIFAR_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms


In [ ]:
# Normalize to mean=0.5, std=0.5 for each channel
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])


In [ ]:
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader

# Load training and test datasets
trainset = CIFAR10(root='./data', train=True, download=True, transform=transform)
testset  = CIFAR10(root='./data', train=False, download=True, transform=transform)

# Split training set into train and val
from torch.utils.data import random_split
val_size = 5000
train_size = len(trainset) - val_size
train_data, val_data = random_split(trainset, [train_size, val_size])


100%|██████████| 170M/170M [00:42<00:00, 3.98MB/s]


In [ ]:
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader

# Load training and test datasets
trainset = CIFAR10(root='./data', train=True, download=True, transform=transform)
testset  = CIFAR10(root='./data', train=False, download=True, transform=transform)

# Split training set into train and val
from torch.utils.data import random_split
val_size = 5000
train_size = len(trainset) - val_size
train_data, val_data = random_split(trainset, [train_size, val_size])


In [ ]:
train_loader = DataLoader(train_data, batch_size=64, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_data,   batch_size=64, shuffle=False, num_workers=2)
test_loader  = DataLoader(testset,    batch_size=64, shuffle=False, num_workers=2)


MLP

In [ ]:
import torch.nn as nn

class MLP(nn.Module):
    def __init__(self, input_dim=3*32*32, hidden_dims=[512, 256], output_dim=10):
        super(MLP, self).__init__()
        layers = []

        dims = [input_dim] + hidden_dims
        for i in range(len(dims) - 1):
            layers.append(nn.Linear(dims[i], dims[i+1]))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(0.3))

        layers.append(nn.Linear(dims[-1], output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        x = x.view(x.size(0), -1)  # Flatten image: [B, 3, 32, 32] → [B, 3072]
        return self.net(x)


In [ ]:
from sklearn.metrics import accuracy_score, f1_score

def train(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(X)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, loader, device):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            out = model(X)
            preds = out.argmax(dim=1)
            y_true.extend(y.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='macro')
    return acc, f1


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MLP().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

for epoch in range(10):  # 10 epochs is usually enough for MLP baseline
    loss = train(model, train_loader, optimizer, criterion, device)
    acc, f1 = evaluate(model, val_loader, device)
    print(f"Epoch {epoch+1} | Train Loss: {loss:.4f} | Val Acc: {acc:.4f} | Val F1: {f1:.4f}")


Epoch 1 | Train Loss: 1.7522 | Val Acc: 0.4336 | Val F1: 0.4267
Epoch 2 | Train Loss: 1.6028 | Val Acc: 0.4532 | Val F1: 0.4473
Epoch 3 | Train Loss: 1.5403 | Val Acc: 0.4700 | Val F1: 0.4643
Epoch 4 | Train Loss: 1.4945 | Val Acc: 0.4786 | Val F1: 0.4739
Epoch 5 | Train Loss: 1.4581 | Val Acc: 0.4942 | Val F1: 0.4907
Epoch 6 | Train Loss: 1.4228 | Val Acc: 0.4972 | Val F1: 0.4932
Epoch 7 | Train Loss: 1.3950 | Val Acc: 0.4964 | Val F1: 0.4967
Epoch 8 | Train Loss: 1.3713 | Val Acc: 0.4994 | Val F1: 0.4983
Epoch 9 | Train Loss: 1.3447 | Val Acc: 0.4964 | Val F1: 0.4943
Epoch 10 | Train Loss: 1.3158 | Val Acc: 0.5108 | Val F1: 0.5107


In [ ]:
test_acc, test_f1 = evaluate(model, test_loader, device)
print(f"\n✅ Test Accuracy: {test_acc:.4f} | F1-score: {test_f1:.4f}")



✅ Test Accuracy: 0.5202 | F1-score: 0.5200


CNN for CIFAR

In [ ]:
import torch.nn as nn

class CIFAR10_CNN(nn.Module):
    def __init__(self):
        super(CIFAR10_CNN, self).__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),  # [B, 3, 32, 32] → [B, 32, 32, 32]
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                         # [B, 32, 16, 16]
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                         # [B, 64, 8, 8]
        )
        self.fc_layers = nn.Sequential(
            nn.Flatten(),                               # [B, 64*8*8]
            nn.Linear(64 * 8 * 8, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 10)                          # 10 classes
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = self.fc_layers(x)
        return x


In [ ]:
def train(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(X)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, loader, device):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            out = model(X)
            preds = out.argmax(dim=1)
            y_true.extend(y.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())
    from sklearn.metrics import accuracy_score, f1_score
    return accuracy_score(y_true, y_pred), f1_score(y_true, y_pred, average='macro')


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CIFAR10_CNN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

for epoch in range(10):
    train_loss = train(model, train_loader, optimizer, criterion, device)
    val_acc, val_f1 = evaluate(model, val_loader, device)
    print(f"Epoch {epoch+1} | Train Loss: {train_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}")


Epoch 1 | Train Loss: 1.4235 | Val Acc: 0.5958 | Val F1: 0.5937
Epoch 2 | Train Loss: 1.0586 | Val Acc: 0.6418 | Val F1: 0.6450
Epoch 3 | Train Loss: 0.8893 | Val Acc: 0.6724 | Val F1: 0.6714
Epoch 4 | Train Loss: 0.7747 | Val Acc: 0.6956 | Val F1: 0.6949
Epoch 5 | Train Loss: 0.6762 | Val Acc: 0.7092 | Val F1: 0.7090
Epoch 6 | Train Loss: 0.5942 | Val Acc: 0.7198 | Val F1: 0.7186
Epoch 7 | Train Loss: 0.5202 | Val Acc: 0.7178 | Val F1: 0.7192
Epoch 8 | Train Loss: 0.4513 | Val Acc: 0.7252 | Val F1: 0.7235
Epoch 9 | Train Loss: 0.3876 | Val Acc: 0.7208 | Val F1: 0.7235
Epoch 10 | Train Loss: 0.3427 | Val Acc: 0.7302 | Val F1: 0.7304


In [ ]:
test_acc, test_f1 = evaluate(model, test_loader, device)
print(f"\n✅ Test Accuracy: {test_acc:.4f} | F1-score: {test_f1:.4f}")



✅ Test Accuracy: 0.7323 | F1-score: 0.7324
